# import libs

In [66]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from tqdm import tqdm


# import data

In [58]:
train_seg1 = pd.read_csv("R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/train_flag_own_car_1.csv")
test_seg1  = pd.read_csv("R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/test_flag_own_car_1.csv")
oot_seg1   = pd.read_csv("R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/oot_flag_own_car_1.csv")

train_seg0 = pd.read_csv("R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/train_flag_own_car_0.csv")
test_seg0 = pd.read_csv("R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/test_flag_own_car_0.csv")
oot_seg0  = pd.read_csv("R:/misunderstand-about-segmentation-and-presegment-pre-modelling/data/oot_flag_own_car_0.csv")

In [59]:
train_seg1_10k = train_seg1.sample(n=10000, random_state=42)
test_seg1_10k = test_seg1.sample(n=8000, random_state=42)
oot_seg1_10k = oot_seg1.sample(n=2000, random_state=42)

train_seg0_10k = train_seg0.sample(n=10000, random_state=42)
test_seg0_10k = test_seg0.sample(n=8000, random_state=42)
oot_seg0_10k = oot_seg0.sample(n=2000, random_state=42)

# statistical test: chow test / interaction test
Trước khi quyết định chia cắt tập dữ liệu thành các sub-models, cần có các kiểm định thống kê để chứng minh sự khác biệt là có thật, tránh rớt vào bẫy "metric chasing" (over-segmentation)


*   **Chow Test:** Kiểm định xem liệu có structural Break giữa các nhóm dữ liệu hay không. Phù hợp để chứng minh các nhóm có hệ số hồi quy thực sự khác biệt
*   **Interaction Test:** Giữ nguyên tập dữ liệu tổng, đưa interaction terms vào mô hình (ví dụ: Logistic Regression) và kiểm tra p-value. Nếu có ý nghĩa thống kê, sự khác biệt giữa các phân khúc mới đáng tin cậy
* **-> just for regression**

# permutation test
idea: nếu segment label không có ý nghĩa, thì shuffle label đi thì model performance không đổi nhiều

- idea: nếu chúng ta tìm ra 1 pattern: nếu segment bằng feature x hoặc tìm ra 1 pattern rằng nếu khách hàng có yếu tố a thì ít default hơn
- experiment: random sample n observations, đều label cho 2 nhóm (làm vậy nhiều lần) -> train riêng model cho mỗi segment -> sau đó train model chung 

In [60]:
# separate

def calc_gini(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    return 2 * auc - 1

def separte_train(train, test, oot, label_col, segment_col):
    X_train = train.drop("target", axis = 1)
    y_train = train["target"]

    X_test = test.drop("target", axis = 1)
    y_test = test["target"]

    X_oot = oot.drop("target", axis = 1)
    y_oot = oot["target"]

    model = XGBClassifier(max_depth = 4)
    model.fit(X_train, y_train)

    gini_train = calc_gini(y_train,model.predict_proba(X_train)[:,1])
    gini_test = calc_gini(y_test,model.predict_proba(X_test)[:,1])
    gini_oot = calc_gini(y_oot,model.predict_proba(X_oot)[:,1])
    
    result = []

    result.append({'segment'    : X_train[segment_col].unique(),
    'gini_train' : round(gini_train, 5),
    'gini_test'  : round(gini_test, 5),
    'gini_oot'   : round(gini_oot, 5),
    'metric_drop': round(1 - gini_oot / gini_train, 5),
    })
    return pd.DataFrame(result)

In [61]:
seg_1_model = separte_train(train_seg1_10k,test_seg1_10k, oot_seg1_10k,'target', 'flag_own_car')

seg_0_model = separte_train(train_seg0_10k,test_seg0_10k, oot_seg0_10k,'target', 'flag_own_car')

n_seg1 = len(oot_seg1_10k)
n_seg0 = len(oot_seg0_10k)
n_total = n_seg1 + n_seg0

gini_separate = (
    seg_1_model['gini_oot'].values[0] * n_seg1 +
    seg_0_model['gini_oot'].values[0] * n_seg0
) / n_total

In [62]:
train = pd.concat([train_seg1_10k, train_seg0_10k])
test = pd.concat([test_seg1_10k, test_seg0_10k])
oot = pd.concat([oot_seg1_10k, oot_seg0_10k])

In [63]:
model = XGBClassifier(max_depth = 2)
model.fit(train.drop('target', axis=1), train['target'])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [64]:
gini_train  = calc_gini(train['target'], model.predict_proba(train.drop('target', axis=1))[:,1])
gini_test   = calc_gini(test['target'],  model.predict_proba(test.drop('target', axis=1))[:,1])
gini_pooled = calc_gini(oot['target'],   model.predict_proba(oot.drop('target', axis=1))[:,1])

In [67]:
delta_obs   = gini_separate - gini_pooled
delta_perm  = []
segment_col = 'flag_own_car'

for i in tqdm(range(500), desc='Permutation test'):
    shuffled_labels = np.random.permutation(train[segment_col].values)

    gini_list = []
    for seg in np.unique(shuffled_labels):
        mask = shuffled_labels == seg
        if mask.sum() < 100:
            continue

        x_seg = train[mask].drop('target', axis=1)
        y_seg = train[mask]['target']

        model.fit(x_seg, y_seg)
        gini_seg = calc_gini(oot['target'],
                             model.predict_proba(oot.drop('target', axis=1))[:,1])
        gini_list.append(gini_seg)

    delta_perm.append(np.mean(gini_list) - gini_pooled)

delta_perm = np.array(delta_perm)
p_value    = np.mean(delta_perm >= delta_obs)

print(f"\nObserved delta : {delta_obs:.4f}")
print(f"p-value        : {p_value:.4f}")
print(f"Significant    : {p_value < 0.05}")

Permutation test: 100%|██████████| 500/500 [02:19<00:00,  3.58it/s]


Observed delta : -0.0873
p-value        : 1.0000
Significant    : False


## Cái bẫy DS hay mắc phải

Nếu một DS thấy kết quả như sau:

| Bước | Hành động |
|---|---|
| 1 | Segment dữ liệu theo `flag_own_car` |
| 2 | Train riêng model cho mỗi nhóm |
| 3 | Gini của từng nhóm **trông cao hơn** trên train set |
| 4 | Kết luận: *"Model tốt hơn sau khi segment!"* |

---

## Thực tế

| Metric | Giá trị |
|---|---|
| Observed delta (separate − pooled) | **-0.087** |
| p-value (permutation test) | **1.000** |
| Significant | **False** |

**Giải thích:**

- Gini trên **OOT thấp hơn** so với pooled model → segmentation làm model tệ hơn
- Permutation test confirm: shuffle label ngẫu nhiên cũng cho kết quả tương tự → không có real signal
- Đây là **AUC inflation do sample homogeneity** — tập nhỏ hơn, đồng nhất hơn → dễ fit hơn → Gini train trông tốt nhưng không generalize
- Không phải genuine improvement

---

## Kết luận

> Segment theo `flag_own_car` **không có ý nghĩa thống kê**.
> Thêm `flag_own_car` vào model như một feature thông thường là đủ —
> không cần và không nên tách thành separate models.

Đây là điểm nhiều DS junior hay mắc phải:
**thấy Gini tăng sau khi segment → vội kết luận model tốt hơn.**
Permutation test là công cụ để tránh bẫy này.


# fundamentally Different Populations
Việc tách riêng model (split dataset) chỉ được biện minh và mang lại giá trị thực sự khi các phân khúc khách hàng xuất phát từ những quần thể có bản chất cốt lõi hoàn toàn khác biệt.

**Ví dụ đúng:** **salaried (đổ lương qua app banking) vs no salary data**
*   *Lý do:* Nguồn dữ liệu (data source), cách chứng minh thu nhập, tính ổn định dòng tiền và hồ sơ tài chính của hai tập khách hàng này hoạt động theo những cơ chế khác nhau hoàn toàn. Do đó, quy luật đánh giá rủi ro cũng sẽ khác biệt.


# correct approach
Thay vì chia nhỏ dữ liệu làm giảm kích thước mẫu (sample size) và khiến mô hình dễ bị overfit trên tập OOT, phương pháp tối ưu trong Machine Learning là:


*   **Segment as a feature (not a split criterion):** 
    Gộp toàn bộ dữ liệu thành một tập train duy nhất. Đưa biến phân khúc vào làm một feature độc lập. Thuật toán (đặc biệt là Tree-based như XGBoost) sẽ tự động tìm ra điểm cắt (split node) tối ưu nếu biến đó thực sự mang lại Information Gain.


*   **Interaction features:** 
    Chủ động tạo ra các biến tương tác để giúp mô hình bắt được hành vi đặc thù của từng nhóm phân khúc trên cùng một tập dữ liệu lớn.
    *   *Ví dụ:* `income_type × payment_to_income`. Biến này giúp mô hình hiểu rằng mức độ rủi ro của tỷ lệ trả nợ/thu nhập sẽ có trọng số khác nhau tùy thuộc vào việc khách hàng làm công ăn lương hay tự doanh.

# conclusion

thêm segment vào model  như 1 feature thay vì tách model
- capture được segment differences
- không bị AUC inflation
- dễ maintain hơn (1 model thay vì N models)
- OOT stable hơn

→ Đây là điểm nhiều DS junior hay mắc phải — thấy AUC tăng sau khi segment là vội kết luận model tốt hơn